## 📝 Problem: Mini Student Exam Score Prediction

### Background: 
You are a data scientist at a small tutoring center. The center wants to know which student factors are most important in predicting exam performance so they can focus their efforts on the right areas.

### Data:

You have 4 student features:

Feature	Description
- X1	Hours studied per day
- X2	Attendance percentage
- X3	Sleep hours per night
- X4	Social media usage hours per day

### The target variable is: 
exam_score → the final exam score (0–100)

The relationship (hidden to you) is:

> exam_score = 5*X1 + 2*X2 + 1.5*X3 - 3*X4 + random noise


### Create synthetic dataset

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 100

#create dataframe
df = pd.DataFrame({
    "hours_studied_per_day": np.random.randint( 2, 7, n),
    "attendence_percentage": np.random.randint(30, 87, n),
    "sleep_hours_per_night": np.random.randint(4, 9, n),
    "social_media_usage": np.random.randint(1, 12, n),
    "random_noise" : np.random.randn(n)
    })

#target
df["exam_score"] = ( 5* df["hours_studied_per_day"] 
                     + 2*df["attendence_percentage"] 
                     + 1.5*df["sleep_hours_per_night"] 
                     - 3*df["social_media_usage"] 
                     +  np.random.randn(n) * 5 )
df.head()

,hours_studied_per_day,attendence_percentage,sleep_hours_per_night,social_media_usage,random_noise,exam_score
0,5,57,4,5,-1.991538,122.796736
1,6,36,7,4,0.426026,101.846588
2,4,38,4,8,-0.541285,87.068333
3,6,37,4,8,0.776823,88.022914
4,6,41,5,7,-0.047644,98.950363


### Step 2: Train and test set

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

X = df.drop(columns=["exam_score"])
y = df["exam_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

feature_names = df.drop(columns=["exam_score"]).columns

### Step 3: Stagewise training (core model)

In [15]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# -----------------------------
# 1. Prepare data
# -----------------------------

X = df.drop(columns=["exam_score"]).values
y = df["exam_score"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

feature_names = df.drop(columns=["exam_score"]).columns

# -----------------------------
# 2. Initialize model
# -----------------------------

feature_strength = np.zeros(X_train.shape[1]) # like "beta"
step_size = 0.01
steps = 3000

# -----------------------------
# 3. Training loop
# -----------------------------

for i in range(steps):

    # predictions using current strengths
    predictions = X_train @ feature_strength

    # mistakes
    mistakes = y_train - predictions

    # find which feature explains mistakes most
    feature_effect = X_train.T @ mistakes

    best_feature = np.argmax(np.abs(feature_effect))

    # decide direction (+ or -)
    direction = np.sign(feature_effect[best_feature])

    # tiny update
    feature_strength[best_feature] += step_size * direction

# -----------------------------
# 4. Evaluation
# -----------------------------

train_pred = X_train @ feature_strength
test_pred = X_test @ feature_strength

print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test MSE:", mean_squared_error(y_test, test_pred))

# -----------------------------
# 5. Final learned weights
# -----------------------------

print("\nLearned feature importance:\n")

for name, value in zip(feature_names, feature_strength):
    print(name, ":", value)

Train MSE: 152.56976828384245
Test MSE: 148.49389078845314

Learned feature importance:

hours_studied_per_day : 0.0
attendence_percentage : 2.209999999999997
sleep_hours_per_night : 0.0
social_media_usage : -0.5700000000000003
random_noise : 0.0
